In [1]:
import os
import pickle
import cv2
import mediapipe as mp
import numpy as np
from tensorflow.keras.models import load_model

VERSION = 4
model = load_model(f'./v{VERSION}/hand_gesture_model.keras')
with open(f'./v{VERSION}/labels.pickle', 'rb') as f:
    labels = pickle.load(f)

cap = cv2.VideoCapture(0)

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.5,
    max_num_hands=2
)

labels_dict = {
    0: 'A', 1: 'B', 2: 'C', 3: 'D',
    4: 'E', 5: 'F', 6: 'G', 7: 'H',
    8: 'I', 9: 'J', 10: 'K', 11: 'L',
    12: 'M', 13: 'N', 14: 'O', 15: 'P',
    16: 'Q', 17: 'R', 18: 'S', 19: 'T',
    20: 'U', 21: 'V', 22: 'W', 23: 'X',
    24: 'Y', 25: 'Z'
}

while True:

    ret, frame = cap.read()
    if not ret:
        print("Cannot read webcam frame")
        break

    H, W, _ = frame.shape
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    if results.multi_hand_landmarks:
        # Store landmarks separately
        hand_data = {
            "Right": None,
            "Left": None
        }

        all_x = []
        all_y = []

        # Extract landmarks and handedness
        for i, hand_landmarks in enumerate(results.multi_hand_landmarks):

            mp_label = results.multi_handedness[i].classification[0].label

            # Raw, unmirrored webcam input uses
            # the opposite handedness label in MediaPipe.
            # Swap to get the actual physical hand.
            hand_label = ("Right" if mp_label == "Left" else "Left")

            coords = []

            for landmark in hand_landmarks.landmark:

                x = landmark.x
                y = landmark.y

                coords.append((x, y))

                all_x.append(x)
                all_y.append(y)

            hand_data[hand_label] = coords

            # Draw landmarks
            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

        # Shared normalization for both hands
        min_x = min(all_x)
        min_y = min(all_y)

        max_x = max(all_x)
        max_y = max(all_y)

        width = max_x - min_x
        height = max_y - min_y

        scale = max(width, height)

        if scale == 0:
            cv2.imshow('frame', frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            continue

        data_aux = []

        # RIGHT HAND FIRST, LEFT HAND SECOND
        for hand_label in ["Right", "Left"]:

            if hand_data[hand_label] is not None:

                for x, y in hand_data[hand_label]:

                    normalized_x = (x - min_x) / scale
                    normalized_y = (y - min_y) / scale

                    data_aux.append(normalized_x)
                    data_aux.append(normalized_y)

            else:
                # Missing hand = 42 zeros
                data_aux.extend([0.0] * 42)

        # Check feature count
        if len(data_aux) == 84:

            prediction = model.predict(np.asarray([data_aux], dtype=np.float32), verbose=0)
            predicted_class = int(np.argmax(prediction[0]))
            predicted_character = labels_dict[predicted_class]

            confidence = float(np.max(prediction[0]))

            # Bounding box
            x1 = max(0, int(min_x * W) - 10)
            y1 = max(0, int(min_y * H) - 10)

            x2 = min(W - 1, int(max_x * W) + 10)
            y2 = min(H - 1, int(max_y * H) + 10)

            cv2.putText(
                frame,
                f'{predicted_character} - {confidence:.2f}',
                (x1, max(30, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.0,
                (255, 255, 255),
                2,
                cv2.LINE_AA
            )

            print(
                f'Prediction: {predicted_character} '
                f'Confidence: {confidence:.2f}'
            )

    cv2.imshow('BISINDO Recognition', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
hands.close()
cv2.destroyAllWindows()


Prediction: A Confidence: 0.83
Prediction: A Confidence: 0.99
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 1.00
Prediction: A Confidence: 0.75
Prediction: B Confidence: 1.00
Prediction: B Confidence: 1.00
Prediction: B Confidence: 0.98
Prediction: B Confidence: 0.99
Prediction: B Confidence: 0.99
Prediction: B Confidence: 0.99
Prediction: B Confidence: 1.00
Prediction: C Confidence: 0.84
Prediction: C Confidence: 0.98
Prediction: C Confidence: 0.99
Prediction: C Confidence: 0.99
Prediction: C Confidence: 0.99
Prediction: C Confidence: 0.99
Prediction: C Confidence: 0.98
Prediction: C Confidence: 0.99
Prediction: P Confidence: 0.62
Prediction: D Confidence: 0.97
Prediction: D Confidence: 0.97
Prediction: D Confidence: 0.98
Prediction: D Confidence: 0.98
Predicti